# Calling LLMs: A Step-by-Step Approach

In this Jupyter notebook, we will explore how to call an LLM through a **LiteLLM** proxy using the OpenAI-compatible API. The model is registered once in LiteLLM (and shows up in Open WebUI), so the notebook only needs the LiteLLM URL and an API key.


## Step 1: Install the Necessary Libraries

In [39]:
!pip install requests

In [40]:
!pip install openai

In [41]:
import requests
import subprocess
import json
from openai import OpenAI

## Step 2: Configure the LiteLLM connection


The notebook runs in the same namespace as LiteLLM, so it can use the internal service URL `http://litellm:4000/v1`. From outside the cluster, use `https://litellm.gccpcai1-ai-application.mindsparks.io/v1` instead. The API key is the LiteLLM master key (or a virtual key created in LiteLLM). Both can be set with the `LITELLM_BASE_URL` and `LITELLM_API_KEY` environment variables.


In [42]:
import os

LITELLM_BASE_URL = os.getenv("LITELLM_BASE_URL", "http://litellm:4000/v1")
LITELLM_API_KEY = os.getenv("LITELLM_API_KEY", "sk-12345678")
MODEL = "qwen3.6-35b"  # model name registered in LiteLLM

client = OpenAI(base_url=LITELLM_BASE_URL, api_key=LITELLM_API_KEY)


Unlike calling the model's inference URL directly, you no longer need the short-lived (10 minute) Unified Analytics token. LiteLLM holds the model credentials and your notebook only uses the LiteLLM key.


## Step 3: List the models available in LiteLLM


This confirms the connection and the key work. The model you registered in LiteLLM should appear in the list.


In [43]:
for m in client.models.list():
    print(m.id)


HTTP Request: GET http://litellm:4000/v1/models "HTTP/1.1 200 OK"


qwen3.6-35b


### NOTE:
*If this fails with 401, the API key does not match the LiteLLM key. If the model is missing, add it in the LiteLLM UI (Models > Add Model).*


## Step 3b (optional): Register the model in LiteLLM from the notebook

Instead of using the LiteLLM UI, the model can be registered with LiteLLM's management API (`/model/new`). Once registered, it also shows up in Open WebUI automatically. This needs `STORE_MODEL_IN_DB=True` on the LiteLLM deployment and the LiteLLM **master key** in `LITELLM_API_KEY`.

The model's `api_key` below is read from the notebook's auth token, which expires after about 10 minutes. Re-run the next cell to refresh it, or replace it with a long-lived token.


In [70]:
LITELLM_ROOT = LITELLM_BASE_URL.rsplit("/v1", 1)[0]
admin_headers = {"Authorization": f"Bearer {LITELLM_API_KEY}", "Content-Type": "application/json"}

with open("/etc/secrets/ezua/.auth_token") as f:
    model_token = f.read().strip()

# (name clients use, model id the server expects, inference service /v1 URL)
MODELS = [
    (
        "qwen3.6-35b",
        "qwen3.6-35b",
        "https://qwen3-6-35b-a3b.project-user-sanjana-upadhyaya-ext.serving.gccpcai1-ai-application.mindsparks.io/v1",
    ),
]


def register_model(model_name, upstream_model, api_base, api_key=model_token):
    # Remove an existing registration with the same name so re-running refreshes the token
    info = requests.get(f"{LITELLM_ROOT}/model/info", headers=admin_headers, verify=False).json()
    for m in info.get("data", []):
        if m["model_name"] == model_name:
            requests.post(f"{LITELLM_ROOT}/model/delete", headers=admin_headers,
                          json={"id": m["model_info"]["id"]}, verify=False)

    payload = {
        "model_name": model_name,
        "litellm_params": {
            "model": f"openai/{upstream_model}",
            "api_base": api_base,
            "api_key": api_key,
            "ssl_verify": False,
            # The model server has no tool-call parser, so it rejects Open WebUI's tool_choice="auto"
            "additional_drop_params": ["tool_choice", "tools"],
        },
        "model_info": {"mode": "chat"},
    }
    r = requests.post(f"{LITELLM_ROOT}/model/new", headers=admin_headers, json=payload, verify=False)
    print(model_name, "->", r.status_code, r.text[:200])


for name, upstream, base in MODELS:
    register_model(name, upstream, base)


qwen3.6-35b -> 200 {"model_id":"97ef2a6e-988d-4e37-a5c8-78d87df86bed","model_name":"qwen3.6-35b","litellm_params":{"model":"HtPoLkOOanF3UUC3nkVSsu8LoDyZkA1hc5S0v1HCP2dlZTDK0lisQPgbwfOgYF_nwW4VhnrgvrMIQg==","api_key":"eP


In [71]:
# Confirm the model is registered and responds through LiteLLM
print([m.id for m in client.models.list()])
test = client.chat.completions.create(model=MODEL_NAME, messages=[{"role": "user", "content": "Say hi"}], max_tokens=16)
print(test.choices[0].message.content)


HTTP Request: GET http://litellm:4000/v1/models "HTTP/1.1 200 OK"


['qwen3.6-35b']


HTTP Request: POST http://litellm:4000/v1/chat/completions "HTTP/1.1 200 OK"


Hi!


## Step 4: Model Interaction

All requests go to the LiteLLM chat endpoint, which works for any model registered in LiteLLM:


In [59]:
endpoint = f"{LITELLM_BASE_URL}/chat/completions"
print(endpoint)


http://litellm:4000/v1/chat/completions


NOTE:
The model itself is registered in LiteLLM with its inference service URL and credentials. The notebook never needs those.


In the following cells, `MODEL` (set in Step 2) is the name registered in LiteLLM:


In [60]:
print(f"Base URL: {LITELLM_BASE_URL}")
print(f"Model:    {MODEL}")


Base URL: http://litellm:4000/v1
Model:    qwen3.6-35b


### Example 1: Using the curl command.

Curl is a command-line tool used to transfer data to or from a server using various protocols, including HTTP. To call LiteLLM you need these headers:

- accept: application/json: You expect a response in JSON format.
- Authorization: Bearer {LITELLM_API_KEY}: Your LiteLLM key.
- Content-Type: application/json: The data you send is JSON.

The body uses the OpenAI chat format, with `model` set to the name registered in LiteLLM.


In [61]:
payload = json.dumps({
    "model": MODEL,
    "messages": [{"role": "user", "content": "Once upon a time"}],
    "max_tokens": 64,
})

curl_command = [
    'curl',
    '-H', 'accept: application/json',
    '-H', f'Authorization: Bearer {LITELLM_API_KEY}',
    '-H', 'Content-Type: application/json',
    '-d', payload,
    '-k',  # Skip SSL verification (only matters for https URLs)
    endpoint
]

result = subprocess.run(curl_command, capture_output=True, text=True)


In [62]:
response = json.loads(result.stdout)

if 'choices' in response and len(response['choices']) > 0:
    print("Response successful.")
    print("-----Generated text:---------------")
    print(response['choices'][0]['message']['content'].strip())
else:
    print("Response not successful.")
    print("Server response:", response)


Response successful.
-----Generated text:---------------
"Once upon a time..." is the classic opening to countless stories! Are you thinking of a fairytale, a novel, or perhaps a specific story you'd like to share or create together?

I'm here to listen, help you brainstorm, or continue the tale if you'd like. 😊


## Example 2: Basic Text Generation. 

We have our API set up so let's explore basic text generation with our LLM. Basic Text Generation in a Large Language Model (LLM) refers to the capability of these models to produce coherent and contextually relevant text based on the input they receive.

In [63]:
headers = {
    "accept": "application/json",
    "Content-Type": "application/json",
    "Authorization": f"Bearer {LITELLM_API_KEY}"
}


In [64]:
def call_api(prompt, max_tokens=500):
    try:
        completion = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.5,
            top_p=0.7,
            max_tokens=max_tokens,
            stream=True
        )

        response_text = ""
        for chunk in completion:
            if chunk.choices and chunk.choices[0].delta.content is not None:
                response_text += chunk.choices[0].delta.content

        return response_text
    except Exception as e:
        return f"Could not get a response from LiteLLM at {LITELLM_BASE_URL}: {e}"


In [65]:
def generate_text(prompt, max_tokens=100):
    """Generate text through LiteLLM.

    Args:
        prompt (str): The input prompt for the LLM.
        max_tokens (int): Maximum number of tokens to generate.

    Returns:
        str: The generated text or an error message.
    """
    data = {
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens
    }

    response = requests.post(endpoint, headers=headers, json=data, verify=False)

    if response.status_code == 200:
        return response.json()["choices"][0]["message"]["content"].strip()
    else:
        return f"Error: {response.status_code} - {response.text}"


In [66]:
# Test the API connection
test_prompt = "Hello world!"
print("---------------------------")
print(generate_text(test_prompt))

---------------------------
Hello! How can I help you today?


## Example 3: Text Summarization

Text summarization is the task of creating a concise and coherent summary of a longer text while preserving its key information. LLMs can be quite effective at this task when given appropriate prompts. Let's try summarizing a paragraph about the Industrial Revolution:

In [67]:
summarization_prompt = """
Summarize the following paragraph in one or two sentences:

The Renaissance, which flourished from the 14th to the 17th centuries, was a period marked by a revival of interest in art, science, and classical antiquity in Europe. Originating in Italy, this cultural movement led to significant developments in painting, literature, and humanism, fostering creativity and innovation that laid the groundwork for the modern world.

Summary:
"""

print("Text Summarization result:")
print("---------------------------")
print(generate_text(summarization_prompt, max_tokens=50))

Text Summarization result:
---------------------------
Flourishing in Europe from the 14th to the 17th centuries, the Renaissance was a cultural movement originating in Italy characterized by a revived interest in art, science, and classical antiquity. This era fostered significant creative and


## Example 4: Question Answering

Question Answeting is the task of to understand and respond to questions posed by users. This involves analyzing the input question, retrieving relevant information from their training data, and generating an accurate and coherent response.

In [68]:
qa_prompt = """
Answer the following question based on the given context:

Context: A small town in the countryside is preparing for its annual harvest festival. The locals are excited about the event, which features a parade, food stalls with local produce, and various competitions like pumpkin carving and pie-eating contests. Everyone is involved in the preparations, from decorating the town square to baking pies and organizing games for children.

Question: What are some traditional activities that townspeople might include in a harvest festival, and how do these activities reflect the community's culture?

Answer:
"""

print("Question Answering result:")
print("---------------------------")
print(generate_text(qa_prompt, max_tokens=50))

Question Answering result:
---------------------------
Based on the context provided, traditional activities for the harvest festival include holding a **parade**, operating **food stalls featuring local produce**, and organizing competitions such as **pumpkin carving** and **pie-eating contests**. The preparations themselves—such


# Example 5: Text simplification

Text simplification is the process of transforming complex or dense text into clearer, more accessible language.

In [69]:
simplification_prompt = """
Simplify the following paragraph by using simpler language and making it easier to understand: .. 
Global warming is a long-term increase in Earth's average surface temperature due to human activities, primarily the burning of fossil fuels, which increases the levels of greenhouse gases in the atmosphere. This phenomenon has far-reaching effects on climate patterns, leading to more severe weather events, rising sea levels, and disruptions to ecosystems.
Simplified Text:

"""
print("Text simplification result:")
print("---------------------------")
print(generate_text(simplification_prompt, max_tokens=50))

Text simplification result:
---------------------------
Global warming is when Earth gets hotter over time because of human actions, especially burning fuels like coal and oil. This puts harmful gases in the air that trap heat. This change causes worse storms, rising oceans, and problems for plants and animals.
